# Module 4: Data Structures — Binary Search Trees

**Prerequisites:** Modules 1–3 (Introduction, Logic & Proofs, Recursion & Induction)

In this module, we explore how to represent and reason about data structures in ACL2. Unlike object-oriented languages, ACL2 encapsulates data using *predicates* (recognizers) rather than classes. We will build up from simple association lists to binary search trees, proving key correctness properties along the way.

## 1. Introduction to Data Structures in ACL2

In languages like Java or Python, data structures are defined using classes with fields and methods. In ACL2, we take a different approach:

- **Data** is represented using lists, cons pairs, and atoms
- **Structure** is enforced by *predicates* (recognizer functions) that check whether a value has the right shape
- **Operations** are ordinary functions that assume their inputs satisfy the recognizer
- **Correctness** is established by *theorems* relating operations to the recognizer

Every data structure in ACL2 follows this pattern:

1. **Representation**: Choose how to represent the data using cons pairs, lists, etc.
2. **Recognizer**: Define a predicate `foo-p` that checks if a value is a valid instance
3. **Accessors**: Define functions to extract components
4. **Operations**: Define functions that manipulate the structure
5. **Theorems**: Prove that operations preserve the recognizer and satisfy key properties

## 2. Association Lists (Alists)

The simplest key-value data structure in ACL2 is the *association list* (alist). An alist is a list of key-value pairs, where each pair is a cons cell `(key . value)`.

| Function | Description |
|----------|-------------|
| `(acons key val alist)` | Add a key-value pair to the front |
| `(assoc key alist)` | Look up a key (returns the pair or nil) |
| `(alistp x)` | Recognizer for alists |
| `(strip-cars alist)` | Extract all keys |
| `(strip-cdrs alist)` | Extract all values |

In [ ]:
; Build a phone book as an alist
(defconst *phone-book*
  (acons 'alice 5551234
    (acons 'bob 5555678
      (acons 'carol 5559999 nil))))

; Look up an entry
(assoc 'bob *phone-book*)

In [ ]:
; Verify it's a valid alist and inspect keys/values
(list (alistp *phone-book*)
      (strip-cars *phone-book*)
      (strip-cdrs *phone-book*))

### Properties of Alists

A fundamental property: looking up a key you just inserted always succeeds:

In [ ]:
; Key property: assoc finds what acons just added
(thm
  (equal (assoc k (acons k v alist))
         (cons k v)))

In [ ]:
; If we look up a different key, acons doesn't interfere
(thm
  (implies (not (equal j k))
           (equal (assoc j (acons k v alist))
                  (assoc j alist))))

### Defining Our Own Alist Operations

In [ ]:
; Remove all pairs with a given key
(defun alist-remove (key alist)
  (cond ((endp alist) nil)
        ((equal key (caar alist))
         (alist-remove key (cdr alist)))
        (t (cons (car alist)
                 (alist-remove key (cdr alist))))))

; After removing key k, looking up a different key j is unaffected
(defthm assoc-of-alist-remove
  (implies (not (equal j k))
           (equal (assoc j (alist-remove k alist))
                  (assoc j alist))))

In [ ]:
; Update: remove old binding, then add new one
(defun alist-update (key val alist)
  (acons key val (alist-remove key alist)))

## 3. Binary Search Trees

Association lists are simple but inefficient: lookup is $O(n)$. For better performance, we use *binary search trees* (BSTs), which provide $O(\log n)$ lookup when balanced.

### Representation

We represent BSTs using nested cons cells:
- `nil` represents an empty BST
- A non-empty BST node is `(key left . right)` where `left` and `right` are BSTs

This is a natural Lisp representation that avoids the overhead of more complex structures.

In [ ]:
; A BST node is (key left . right) where left and right are BSTs
; nil represents an empty BST
(defun bst-key (node) (car node))
(defun bst-left (node) (cadr node))
(defun bst-right (node) (cddr node))

In [ ]:
; Recognizer for a BST node (non-nil node with a key)
(defun bst-node-p (x)
  (and (consp x)
       (consp (cdr x))))

### The BST Ordering Property

A binary search tree must satisfy the *BST property*: for every node with key $k$,
- All keys in the left subtree are $< k$
- All keys in the right subtree are $\geq k$

We use `lexorder` for comparisons and helper predicates for bounding:

In [ ]:
; All keys in tree are less than bound
(defun bst-all-< (tree bound)
  (if (null tree)
      t
    (and (lexorder (bst-key tree) bound)
         (not (equal (bst-key tree) bound))
         (bst-all-< (bst-left tree) bound)
         (bst-all-< (bst-right tree) bound))))

; All keys in tree are greater than or equal to bound
(defun bst-all->= (tree bound)
  (if (null tree)
      t
    (and (lexorder bound (bst-key tree))
         (bst-all->= (bst-left tree) bound)
         (bst-all->= (bst-right tree) bound))))

In [ ]:
; The BST property: ordered binary tree
(defun bst-p (tree)
  (if (null tree)
      t
    (and (bst-node-p tree)
         (bst-all-< (bst-left tree) (bst-key tree))
         (bst-all->= (bst-right tree) (bst-key tree))
         (bst-p (bst-left tree))
         (bst-p (bst-right tree)))))

### BST Lookup and Insertion

To search for a key, compare with the current node and recurse into the appropriate subtree. To insert, walk down to the correct position and create a new leaf:

In [ ]:
; Lookup: is key k in the BST?
(defun bst-in (k tree)
  (if (null tree)
      nil
    (cond ((equal k (bst-key tree)) t)
          ((lexorder k (bst-key tree))
           (bst-in k (bst-left tree)))
          (t (bst-in k (bst-right tree))))))

In [ ]:
; Insert key k into BST
(defun bst-insert (k tree)
  (if (null tree)
      (cons k (cons nil nil))  ; new leaf: (k nil . nil)
    (cond ((equal k (bst-key tree)) tree)  ; already present
          ((lexorder k (bst-key tree))
           (cons (bst-key tree)
                 (cons (bst-insert k (bst-left tree))
                       (bst-right tree))))
          (t
           (cons (bst-key tree)
                 (cons (bst-left tree)
                       (bst-insert k (bst-right tree))))))))

### Testing Our BST Operations

In [ ]:
; Build a BST by inserting several keys
(defconst *bst-example*
  (bst-insert 3
    (bst-insert 1
      (bst-insert 5
        (bst-insert 2
          (bst-insert 4 nil))))))

; Test lookup and BST property
(list (bst-in 3 *bst-example*)   ; T
      (bst-in 7 *bst-example*)   ; NIL
      (bst-in 1 *bst-example*)   ; T
      (bst-p *bst-example*))     ; T

### Correctness Theorems for BSTs

We prove the two most important properties:
1. **Preservation**: `bst-insert` preserves the BST property
2. **Completeness**: `bst-in` finds what was inserted

In [ ]:
; Helper lemmas: insertion preserves bounds
(defthm bst-all-<-of-insert
  (implies (and (bst-all-< tree bound)
                (lexorder k bound)
                (not (equal k bound)))
           (bst-all-< (bst-insert k tree) bound)))

(defthm bst-all->=-of-insert
  (implies (and (bst-all->= tree bound)
                (lexorder bound k))
           (bst-all->= (bst-insert k tree) bound)))

In [ ]:
; Main theorem: insertion preserves the BST property
(defthm bst-p-of-bst-insert
  (implies (bst-p tree)
           (bst-p (bst-insert k tree))))

In [ ]:
; After inserting k, we can find k
(defthm bst-in-of-bst-insert-same
  (implies (bst-p tree)
           (bst-in k (bst-insert k tree))))

; Inserting k doesn't affect lookup of other keys
(defthm bst-in-of-bst-insert-other
  (implies (and (bst-p tree)
                (not (equal j k)))
           (equal (bst-in j (bst-insert k tree))
                  (bst-in j tree))))

## 4. Balanced Binary Search Trees

An unbalanced BST can degenerate into a linked list, making operations $O(n)$. For example, inserting $1, 2, 3, 4, 5$ in order produces a right-leaning chain. To guarantee $O(\log n)$ operations, we need *balanced* BSTs.

### Leftist Trees

The ACL2 community books include a verified implementation of *leftist trees* in `books/projects/leftist-trees/`. A leftist tree satisfies:
1. **Heap property**: parent key $\leq$ children keys
2. **Leftist property**: rank of left child $\geq$ rank of right child

where the *rank* is the length of the rightmost path to nil. The leftist property ensures the right spine is short ($O(\log n)$), giving logarithmic merge operations.

| Operation | Balanced BST | Unbalanced BST |
|-----------|-------------|----------------|
| Lookup    | $O(\log n)$ | $O(n)$        |
| Insert    | $O(\log n)$ | $O(n)$        |
| Delete    | $O(\log n)$ | $O(n)$        |

In [ ]:
; Rank of a leftist tree (length of right spine)
(defun ltree-rank (tree)
  (if (null tree)
      0
    (+ 1 (ltree-rank (cddr tree)))))

## 5. Records and Structures

For modeling real-world data, we model records using alists with named fields:

In [ ]:
; A student record as an alist
(defun make-student (name id gpa)
  (list (cons 'name name)
        (cons 'id id)
        (cons 'gpa gpa)))

(defun student-name (s) (cdr (assoc 'name s)))
(defun student-id (s) (cdr (assoc 'id s)))
(defun student-gpa (s) (cdr (assoc 'gpa s)))

; Test
(defconst *alice* (make-student 'alice 12345 39/10))
(list (student-name *alice*) (student-gpa *alice*))

### Modeling a Stack and Queue

In [ ]:
; Stack operations
(defun stack-push (x stk) (cons x stk))
(defun stack-pop (stk) (cdr stk))
(defun stack-top (stk) (car stk))
(defun stack-empty-p (stk) (endp stk))

; Fundamental stack properties
(defthm stack-top-of-push
  (equal (stack-top (stack-push x stk)) x))

(defthm stack-pop-of-push
  (equal (stack-pop (stack-push x stk)) stk))

In [ ]:
; Queue using two lists for amortized O(1) operations
(defun make-queue () (cons nil nil))

(defun enqueue (x q)
  (cons (car q) (cons x (cdr q))))

(defun queue-flip (q)
  (if (consp (car q))
      q
    (cons (reverse (cdr q)) nil)))

(defun dequeue (q)
  (let ((q2 (queue-flip q)))
    (cons (cdar q2) (cdr q2))))

(defun queue-front (q)
  (let ((q2 (queue-flip q)))
    (caar q2)))

(defun queue-empty-p (q)
  (and (endp (car q)) (endp (cdr q))))

In [ ]:
; Test the queue
(let* ((q (make-queue))
       (q (enqueue 'a q))
       (q (enqueue 'b q))
       (q (enqueue 'c q)))
  (list (queue-front q)
        (queue-empty-p q)))

## 6. Exercises

### Exercise 4.1: Alist Merge

Define a function `(alist-merge a1 a2)` that merges two alists, with keys in `a1` taking precedence over keys in `a2`. Prove that for any key `k`, `(assoc k (alist-merge a1 a2))` equals `(assoc k a1)` when present in `a1`, and `(assoc k a2)` otherwise.

In [ ]:
; Exercise 4.1: Define alist-merge and prove its correctness
; YOUR CODE HERE


### Exercise 4.2: BST Minimum

Define a function `(bst-min tree)` that returns the minimum key in a non-empty BST. Prove that the minimum is always found in the leftmost path.

*Hint:* In a BST, the minimum is the leftmost leaf.

In [ ]:
; Exercise 4.2: Define bst-min and prove its properties
; YOUR CODE HERE


### Exercise 4.3: BST Deletion

Define a function `(bst-delete k tree)` that removes key `k` from a BST while maintaining the BST property. You will need a helper to find the minimum of the right subtree when the deleted node has two children.

Prove:
1. `(bst-p (bst-delete k tree))` when `(bst-p tree)`
2. `(not (bst-in k (bst-delete k tree)))` when `(bst-p tree)`

In [ ]:
; Exercise 4.3: Define bst-delete and prove its properties
; YOUR CODE HERE


### Exercise 4.4: BST to Sorted List

Define a function `(bst-to-list tree)` that performs an in-order traversal and returns a sorted list of all keys. Prove that the result is sorted (each element is $\leq$ the next).

In [ ]:
; Exercise 4.4: Define bst-to-list and prove the output is sorted
; YOUR CODE HERE


## Summary

In this module, we learned:

- **Alists** provide simple key-value storage with $O(n)$ lookup
- **Binary search trees** improve lookup to $O(\log n)$ when balanced
- We can *prove* that BST operations preserve structural invariants
- **Balanced trees** (like leftist trees) guarantee logarithmic performance
- **Records, stacks, and queues** can all be modeled with cons-based structures
- Every data structure follows the pattern: representation → recognizer → operations → theorems

The key insight is that in ACL2, data structure correctness is not just tested but *formally verified*.

---

**Next Module:** [Module 5: Graph Algorithms](05_graph_algorithms.ipynb) — We apply these data structure techniques to represent and reason about graphs, including DFS, BFS, and Dijkstra's shortest path algorithm.